# Jalon 1 & 2 — Data & EDA

## Jalon 1 : Choix et justification du dataset

**Dataset** : *TikTok & Instagram Addiction Dataset (2015–2060)* — Kaggle / Abdul Malik Lodhra  
**Licence** : CC BY-NC-SA 4.0  

### Pourquoi ce dataset ?
| Contrainte du sujet | Ce dataset |
|---|---|
| Tâche supervisée claire | `addiction_score` continu (régression) ou `addiction_level` (classification 3 classes) |
| Adapté au DL fondamental (MLP) | 19 features numériques/catégoriques tabulaires → MLP naturel |
| Adapté au DL avancé | Série temporelle 2015–2060 par pays → LSTM/Transformer |
| Volume suffisant | 10 000 lignes (fichier principal) + 50 000 (comportements) |
| Pas dans le repo Git | Dataset exclu via `.gitignore` |

**Tâche principale choisie** : prédiction de `addiction_score` (régression, 0–100).  
Ce choix permet d'utiliser Ridge/Lasso/ElasticNet nativement (régularisation L2/L1) et de faire du forecasting LSTM.

---

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.preprocessing import load_data, FEATURES, TARGET

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110

df = load_data('../data/tiktok_instagram_global_100countries.csv')
print(f'Dimensions : {df.shape}')
df.head(3)

## Jalon 2 — EDA
### 2.1 Aperçu général, types et valeurs manquantes

In [ ]:
print('=== Types de données ===')
print(df.dtypes)
print('\n=== Valeurs manquantes ===')
missing = df.isnull().sum()
print(missing[missing > 0] if missing.any() else 'Aucune valeur manquante')
print(f'\nDoublons : {df.duplicated().sum()}')

In [ ]:
df.describe().round(2)

### 2.2 Distribution de la variable cible

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(df['addiction_score'], bins=40, edgecolor='black', alpha=0.8)
axes[0].set_title('Distribution de addiction_score')
axes[0].set_xlabel('Score (0–100)')
axes[0].axvline(df['addiction_score'].mean(), color='red', linestyle='--', label=f'Moyenne : {df["addiction_score"].mean():.1f}')
axes[0].legend()

df['addiction_level'].value_counts().plot.bar(ax=axes[1], edgecolor='black', alpha=0.8)
axes[1].set_title('Distribution de addiction_level (classes)')
axes[1].set_xlabel('Niveau')
axes[1].set_ylabel('Effectif')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

### 2.3 Distributions des features numériques

In [ ]:
num_cols = [c for c in FEATURES if c != 'country_enc']
n = len(num_cols)
ncols = 4
nrows = (n + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(16, nrows * 3))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    axes[i].hist(df[col], bins=30, edgecolor='black', alpha=0.75)
    axes[i].set_title(col, fontsize=9)
    axes[i].tick_params(labelsize=7)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Distributions des features numériques', y=1.01)
plt.tight_layout()
plt.show()

### 2.4 Matrice de corrélation

In [ ]:
corr_cols = num_cols + ['addiction_score']
corr_matrix = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(13, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f',
    cmap='RdYlGn', center=0, linewidths=0.5, ax=ax, annot_kws={'size': 7}
)
ax.set_title('Matrice de corrélation (features + cible)', fontsize=13)
plt.tight_layout()
plt.show()

print('\nTop corrélations avec addiction_score :')
print(corr_matrix['addiction_score'].drop('addiction_score').abs().sort_values(ascending=False).head(8))

### 2.5 Tendances temporelles (2015–2060)

In [ ]:
yearly = df.groupby('year')[['addiction_score', 'attention_span_score', 'tiktok_minutes_daily', 'instagram_minutes_daily']].mean()

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
cols_plot = {
    'addiction_score': 'Addiction Score moyen',
    'attention_span_score': 'Attention Span Score moyen',
    'tiktok_minutes_daily': 'TikTok minutes/jour (moy.)',
    'instagram_minutes_daily': 'Instagram minutes/jour (moy.)'
}
for ax, (col, title) in zip(axes.flatten(), cols_plot.items()):
    ax.plot(yearly.index, yearly[col], marker='o', markersize=3)
    ax.set_title(title)
    ax.set_xlabel('Année')
    ax.grid(True, alpha=0.3)

plt.suptitle('Évolution temporelle 2015–2060 (moyennes mondiales)', fontsize=13)
plt.tight_layout()
plt.show()

### 2.6 Analyse par pays — Top 15 pays par addiction

In [ ]:
country_mean = df.groupby('country')['addiction_score'].mean().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

country_mean.head(15).plot.barh(ax=axes[0], color='salmon', edgecolor='black')
axes[0].set_title('Top 15 pays — addiction_score moyen')
axes[0].invert_yaxis()
axes[0].set_xlabel('Score moyen')

country_mean.tail(15).plot.barh(ax=axes[1], color='steelblue', edgecolor='black')
axes[1].set_title('Bottom 15 pays — addiction_score moyen')
axes[1].invert_yaxis()
axes[1].set_xlabel('Score moyen')

plt.tight_layout()
plt.show()

### 2.7 Détection des anomalies (méthode IQR)

In [ ]:
def iqr_outliers(series):
    Q1, Q3 = series.quantile(0.25), series.quantile(0.75)
    IQR = Q3 - Q1
    return ((series < Q1 - 1.5 * IQR) | (series > Q3 + 1.5 * IQR)).sum()

outlier_counts = {col: iqr_outliers(df[col]) for col in num_cols}
outlier_df = pd.Series(outlier_counts).sort_values(ascending=False)
print('Nombre de valeurs aberrantes par feature (méthode IQR) :')
print(outlier_df[outlier_df > 0])

fig, ax = plt.subplots(figsize=(12, 4))
df[num_cols[:10]].boxplot(ax=ax, vert=True)
ax.set_title('Boxplots — détection des outliers (10 premières features)')
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

### 2.8 Analyse par groupe d'âge (dataset screen_time)

Le fichier `screen_time_behavior.csv` (50 000 lignes) apporte une dimension démographique complémentaire.

In [ ]:
df_screen = pd.read_csv('../data/screen_time_behavior.csv')
print(f'Dimensions screen_time : {df_screen.shape}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df_screen.groupby('age_group')['weekday_screen_hours'].mean().sort_values().plot.bar(
    ax=axes[0], edgecolor='black', alpha=0.8
)
axes[0].set_title('Heures écran (semaine) par groupe d\'âge')
axes[0].tick_params(axis='x', rotation=30)

df_screen.groupby('platform')['focus_span_minutes'].mean().sort_values().plot.barh(
    ax=axes[1], edgecolor='black', alpha=0.8
)
axes[1].set_title('Focus span moyen par plateforme')

plt.tight_layout()
plt.show()

## Synthèse EDA

| Observation | Impact sur la modélisation |
|---|---|
| Pas de valeurs manquantes | Pas de stratégie d'imputation nécessaire |
| `addiction_score` distribué sur toute la plage 0–100 | Tâche de régression directe |
| Forte corrélation `tiktok_minutes_daily` ↔ `addiction_score` | Feature importante pour le modèle |
| Tendance haussière 2015→2060 | Structure temporelle exploitable par LSTM |
| Dataset synthétique | Résultats de haute performance attendus |
| Peu d'outliers significatifs | StandardScaler suffisant, pas de winsorisation nécessaire |